In [2]:
# installing libraries
!pip uninstall -y numpy tensorflow tensorflow_cpu tensorflow-intel keras
!pip install numpy==2.0.2 pandas==2.2.3 tensorflow==2.18.0

Found existing installation: numpy 2.3.1
Uninstalling numpy-2.3.1:
  Successfully uninstalled numpy-2.3.1


     ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
     - -------------------------------------- 0.8/18.9 MB 15.4 MB/s eta 0:00:02
     ------------------- -------------------- 9.2/18.9 MB 34.7 MB/s eta 0:00:01
     -------------------------------- ------ 15.7/18.9 MB 33.7 MB/s eta 0:00:01
     ---------------------------------------- 18.9/18.9 MB 28.3 MB/s  0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'error'


  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [19 lines of output]
      + C:\Users\helia\AppData\Local\Programs\Python\Python313\python.exe C:\Users\helia\AppData\Local\Temp\pip-install-5a9aeqs1\numpy_992180dc74cd4c918d7c096a4f46b353\vendored-meson\meson\meson.py setup C:\Users\helia\AppData\Local\Temp\pip-install-5a9aeqs1\numpy_992180dc74cd4c918d7c096a4f46b353 C:\Users\helia\AppData\Local\Temp\pip-install-5a9aeqs1\numpy_992180dc74cd4c918d7c096a4f46b353\.mesonpy-ao61t74x -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --native-file=C:\Users\helia\AppData\Local\Temp\pip-install-5a9aeqs1\numpy_992180dc74cd4c918d7c096a4f46b353\.mesonpy-ao61t74x\meson-python-native-file.ini
      The Meson build system
      Version: 1.4.99
      Source dir: C:\Users\helia\AppData\Local\Temp\pip-install-5a9aeqs1\numpy_992180dc74cd4c918d7c096a4f46b353
      Build dir: C:\Users\helia\AppData\Local\Temp\pip-install-5a9

In [3]:
import tensorflow as tf
print(tf.__version__)

2.18.0


In [4]:
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

In [5]:
import pandas as pd
import numpy as np
import keras

import warnings
warnings.simplefilter('ignore', FutureWarning)  

In [6]:
# load data
filepath='https://s3-api.us-geo.objectstorage.softlayer.net/cf-courses-data/CognitiveClass/DL0101EN/labs/data/concrete_data.csv'
concrete_data = pd.read_csv(filepath)

concrete_data.head()

,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age,Strength
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.99
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.89
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.27
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.05
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.30


The table shows that a 28-day-old concrete mix with a compressive strength of 79.99 MPa contains 540 cubic meters of cement, 162 cubic meters of water, 2.5 cubic meters of superplasticizer, 1040 cubic meters of coarse aggregate, and 676 cubic meters of fine aggregate, while containing no blast furnace slag or fly ash.


In [7]:
concrete_data.shape

(1030, 9)

In [8]:
# checking data for missing values
concrete_data.describe()

,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age,Strength
count,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000
mean,281.167864,73.895825,54.188350,181.567282,6.204660,972.918932,773.580485,45.662136,35.817961
std,104.506364,86.279342,63.997004,21.354219,5.973841,77.753954,80.175980,63.169912,16.705742
min,102.000000,0.000000,0.000000,121.800000,0.000000,801.000000,594.000000,1.000000,2.330000
25%,192.375000,0.000000,0.000000,164.900000,0.000000,932.000000,730.950000,7.000000,23.710000
50%,272.900000,22.000000,0.000000,185.000000,6.400000,968.000000,779.500000,28.000000,34.445000
75%,350.000000,142.950000,118.300000,192.000000,10.200000,1029.400000,824.000000,56.000000,46.135000
max,540.000000,359.400000,200.100000,247.000000,32.200000,1145.000000,992.600000,365.000000,82.600000


In [ ]:
concrete_data.isnull().sum()
# the data will look clean and ready

Cement                0
Blast Furnace Slag    0
Fly Ash               0
Water                 0
Superplasticizer      0
Coarse Aggregate      0
Fine Aggregate        0
Age                   0
Strength              0
dtype: int64

splitting data into predictors and target

In [11]:
concrete_data_columns = concrete_data.columns

predictors = concrete_data[concrete_data_columns[concrete_data_columns != 'Strength']] # all columns except Strength
target = concrete_data['Strength'] # Strength column

In [12]:
# check if we split correctly
predictors.head()

,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360


In [13]:
target.head()

0    79.99
1    61.89
2    40.27
3    41.05
4    44.30
Name: Strength, dtype: float64

normalizing data

In [14]:
predictors_norm = (predictors - predictors.mean()) / predictors.std()
predictors_norm.head()

,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age
0,2.476712,-0.856472,-0.846733,-0.916319,-0.620147,0.862735,-1.217079,-0.279597
1,2.476712,-0.856472,-0.846733,-0.916319,-0.620147,1.055651,-1.217079,-0.279597
2,0.491187,0.795140,-0.846733,2.174405,-1.038638,-0.526262,-2.239829,3.551340
3,0.491187,0.795140,-0.846733,2.174405,-1.038638,-0.526262,-2.239829,5.055221
4,-0.790075,0.678079,-0.846733,0.488555,-1.038638,0.070492,0.647569,4.976069


In [17]:
n_cols = predictors_norm.shape[1] # number of predictors

In [15]:
# importing keras packages
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Input

In [ ]:
# build a neural network
# define regression model
def regression_model():
    # create model
    model = Sequential()
    model.add(Input(shape=(n_cols,)))
    model.add(Dense(50, activation='relu'))
    model.add(Dense(50, activation='relu'))
    model.add(Dense(1)) # output node
    
    # compile model
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

In [20]:
# test and train

#building the model
model = regression_model()

# train and test with fit method
model.fit(predictors_norm, target, validation_split=0.3, epochs=100, verbose=2)

Epoch 1/100
23/23 - 1s - 47ms/step - loss: 1646.9209 - val_loss: 1142.6704
Epoch 2/100
23/23 - 0s - 14ms/step - loss: 1512.5255 - val_loss: 1022.0510
Epoch 3/100
23/23 - 0s - 10ms/step - loss: 1289.1271 - val_loss: 826.5166
Epoch 4/100
23/23 - 0s - 8ms/step - loss: 948.5624 - val_loss: 574.9525
Epoch 5/100
23/23 - 0s - 10ms/step - loss: 574.8063 - val_loss: 343.6696
Epoch 6/100
23/23 - 0s - 7ms/step - loss: 325.2027 - val_loss: 220.2589
Epoch 7/100
23/23 - 0s - 7ms/step - loss: 246.6055 - val_loss: 183.7189
Epoch 8/100
23/23 - 0s - 12ms/step - loss: 223.0253 - val_loss: 172.4520
Epoch 9/100
23/23 - 0s - 9ms/step - loss: 207.1314 - val_loss: 168.5457
Epoch 10/100
23/23 - 0s - 7ms/step - loss: 195.9284 - val_loss: 165.1560
Epoch 11/100
23/23 - 0s - 8ms/step - loss: 187.9579 - val_loss: 164.5197
Epoch 12/100
23/23 - 0s - 8ms/step - loss: 181.3164 - val_loss: 162.4892
Epoch 13/100
23/23 - 0s - 10ms/step - loss: 175.6010 - val_loss: 161.5093
Epoch 14/100
23/23 - 0s - 11ms/step - loss: 170.6

Changing model parameters

In [ ]:
def regression_model():
    input_colm = predictors_norm.shape[1] # number of input features
    # create model
    model = Sequential()
    model.add(Input(shape=(input_colm,)))  # Set the number of input features 
    model.add(Dense(50, activation='relu'))  
    model.add(Dense(50, activation='relu'))
    model.add(Dense(50, activation='relu')) 
    model.add(Dense(50, activation='relu'))
    model.add(Dense(50, activation='relu'))  
    model.add(Dense(1))  # Output layer
    
    # compile model
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

In [22]:
# build the model
model = regression_model()
model.fit(predictors_norm, target, validation_split=0.1, epochs=100, verbose=2)

Epoch 1/100
29/29 - 3s - 113ms/step - loss: 1521.5531 - val_loss: 1001.3141
Epoch 2/100
29/29 - 0s - 12ms/step - loss: 620.5552 - val_loss: 232.9743
Epoch 3/100
29/29 - 0s - 6ms/step - loss: 245.2923 - val_loss: 229.0022
Epoch 4/100
29/29 - 0s - 6ms/step - loss: 207.6233 - val_loss: 195.8260
Epoch 5/100
29/29 - 0s - 8ms/step - loss: 184.4243 - val_loss: 181.6415
Epoch 6/100
29/29 - 0s - 7ms/step - loss: 172.0542 - val_loss: 166.0230
Epoch 7/100
29/29 - 0s - 7ms/step - loss: 159.8221 - val_loss: 155.3120
Epoch 8/100
29/29 - 0s - 8ms/step - loss: 147.9654 - val_loss: 139.0696
Epoch 9/100
29/29 - 0s - 7ms/step - loss: 137.2723 - val_loss: 138.2600
Epoch 10/100
29/29 - 0s - 7ms/step - loss: 127.7015 - val_loss: 126.1037
Epoch 11/100
29/29 - 0s - 7ms/step - loss: 112.6357 - val_loss: 108.9292
Epoch 12/100
29/29 - 0s - 8ms/step - loss: 96.3583 - val_loss: 93.0380
Epoch 13/100
29/29 - 0s - 9ms/step - loss: 81.8692 - val_loss: 92.5873
Epoch 14/100
29/29 - 0s - 9ms/step - loss: 71.3015 - val_lo

Increasing the number of hidden layers gives the model greater ability to capture complex patterns in the data, which can improve how well it fits the training set and, in turn, strengthen its predictions.

Using a smaller validation split leaves more data available for training, giving the model more examples from which to learn. With access to a larger training set, it can better recognize underlying trends and potentially achieve stronger overall performance.
